# Car Price Prediction — Linear, Ridge and Lasso Regression
**Simple assignment notebook | 60 minutes**

**Goal:** Predict the continuous target `price` and compare Linear Regression, Ridge and Lasso.

**Dataset:** Kaggle — `hellbuoy/car-price-prediction`  
Download **`CarPrice_Assignment.csv`** and upload it to Colab (or keep it beside this notebook). Run cells from top to bottom.


## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load dataset

In [ ]:
df = pd.read_csv('CarPrice_Assignment.csv')
df.head()

In [ ]:
print(df.shape)
print(df.isnull().sum())

## 3. Choose features and target

To keep the code easy, we use **numeric car features only**. We do not use `car_ID` (just an ID) or `CarName` (text). The target is `price`.


In [ ]:
features = ['wheelbase', 'carwidth', 'curbweight',
            'enginesize', 'horsepower', 'citympg', 'highwaympg']

X = df[features]
y = df['price']

X.head()

## 4. Train/test split (80% training, 20% testing)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

## 5. Scale features

Scaling helps Ridge and Lasso compare features fairly. Fit the scaler **only on training data**.


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 6. Train all three models

In [ ]:
linear = LinearRegression()
ridge = Ridge(alpha=10)
lasso = Lasso(alpha=100, max_iter=100000)

linear.fit(X_train, y_train)
ridge.fit(X_train, y_train)
lasso.fit(X_train, y_train)

## 7. Compare MAE, MSE, RMSE and R² on the test set

In [ ]:
models = {'Linear Regression': linear, 'Ridge': ridge, 'Lasso': lasso}

results = []

for name, model in models.items():
    prediction = model.predict(X_test)
    mae = mean_absolute_error(y_test, prediction)
    mse = mean_squared_error(y_test, prediction)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, prediction)
    results.append([name, mae, mse, rmse, r2])

comparison = pd.DataFrame(results, columns=['Model', 'MAE', 'MSE', 'RMSE', 'R2'])
comparison.round(2)

## 8. Compare coefficients

In [ ]:
coefficients = pd.DataFrame({
    'Feature': features,
    'Linear': linear.coef_,
    'Ridge': ridge.coef_,
    'Lasso': lasso.coef_
})
coefficients.round(2)

**What the coefficients mean:** Because the features are standardized, each coefficient represents the approximate price change for a one-standard-deviation increase in that feature, holding other included features fixed. Ridge usually shrinks coefficients; Lasso can set some coefficients to zero.


## 9. Conclusion (automatically based on your results)

In [ ]:
best = comparison.loc[comparison['RMSE'].idxmin(), 'Model']
print('Best model on this test split:', best)
print('Ridge shrinks coefficients; Lasso may reduce some coefficients to zero.')
print('Lower MAE, MSE, RMSE and higher R² indicate better test performance.')
print('The best test model generalizes best on this split, but one split is not a guarantee.')

### Short written conclusion (2–3 sentences)

After running the notebook, use the printed result to complete this conclusion:

> **[Best model]** performed best on the held-out test data because it had the lowest RMSE among the three models. Ridge and Lasso use regularization to reduce coefficient sizes, which can help limit overfitting; Lasso can also set some coefficients to zero. Since this result comes from one train/test split, cross-validation would provide stronger evidence of generalization.

**Note:** The notebook does not invent scores before the dataset is loaded. All metrics and the best-model label are calculated when you run it.
